# Chapter 15: deep learning on unseen data: introducing federated learning

Haikal Ali TK-46-GAB 1103223071

## In this chapter

- The problem of privacy in deep learning
- Federated learning
- Learning to detect spam
- Hacking into federated learning
- Secure aggregation
- Homomorphic encryption
- Homomorphically encrypted federated learning

## Setup framework dependency

Chapter ini memakai framework kecil dari chapter sebelumnya: `Tensor`, autograd, `SGD`, `Embedding`, dan `MSELoss`. Cell berikut disertakan agar notebook ini bisa langsung dijalankan di Colab tanpa harus menjalankan notebook chapter sebelumnya.

In [1]:
import copy
import sys
import random
import codecs
from collections import Counter
from pathlib import Path

import numpy as np

np.random.seed(12345)

class Tensor(object):
  def __init__(
    self,
    data,
    autograd=False,
    creators=None,
    creation_op=None,
    id=None
  ):
    self.data = np.array(data)
    self.creators = creators
    self.creation_op = creation_op
    self.grad = None
    self.autograd = autograd
    self.children = {}

    if id is None:
      id = np.random.randint(0, 100000)

    self.id = id

    if creators is not None:
      for c in creators:
        if self.id not in c.children:
          c.children[self.id] = 1
        else:
          c.children[self.id] += 1

  def all_children_grads_accounted_for(self):
    for id, cnt in self.children.items():
      if cnt != 0:
        return False

    return True

  def backward(self, grad=None, grad_origin=None):
    if not self.autograd:
      return

    if grad is None:
      grad = Tensor(np.ones_like(self.data))

    if grad_origin is not None:
      if self.children[grad_origin.id] == 0:
        raise Exception("cannot backprop more than once")
      else:
        self.children[grad_origin.id] -= 1

    if self.grad is None:
      self.grad = grad
    else:
      self.grad += grad

    if self.creators is not None and (
      self.all_children_grads_accounted_for() or grad_origin is None
    ):
      if self.creation_op == "add":
        self.creators[0].backward(self.grad, self)
        self.creators[1].backward(self.grad, self)

      if self.creation_op == "neg":
        self.creators[0].backward(self.grad.__neg__(), self)

      if self.creation_op == "sub":
        self.creators[0].backward(Tensor(self.grad.data), self)
        self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

      if self.creation_op == "mul":
        self.creators[0].backward(self.grad * self.creators[1], self)
        self.creators[1].backward(self.grad * self.creators[0], self)

      if self.creation_op is not None and "sum" in self.creation_op:
        dim = int(self.creation_op.split("_")[1])
        ds = self.creators[0].data.shape[dim]
        self.creators[0].backward(self.grad.expand(dim, ds), self)

      if self.creation_op is not None and "expand" in self.creation_op:
        dim = int(self.creation_op.split("_")[1])
        self.creators[0].backward(self.grad.sum(dim), self)

      if self.creation_op == "sigmoid":
        ones = Tensor(np.ones_like(self.grad.data))
        self.creators[0].backward(self.grad * (self * (ones - self)), self)

      if self.creation_op == "index_select":
        new_grad = np.zeros_like(self.creators[0].data)
        indices_ = self.index_select_indices.data.flatten()
        grad_ = self.grad.data.reshape(len(indices_), -1)

        for i in range(len(indices_)):
          new_grad[indices_[i]] += grad_[i]

        self.creators[0].backward(Tensor(new_grad), self)

  def __add__(self, other):
    if self.autograd and other.autograd:
      return Tensor(
        self.data + other.data,
        autograd=True,
        creators=[self, other],
        creation_op="add"
      )

    return Tensor(self.data + other.data)

  def __neg__(self):
    if self.autograd:
      return Tensor(
        self.data * -1,
        autograd=True,
        creators=[self],
        creation_op="neg"
      )

    return Tensor(self.data * -1)

  def __sub__(self, other):
    if self.autograd and other.autograd:
      return Tensor(
        self.data - other.data,
        autograd=True,
        creators=[self, other],
        creation_op="sub"
      )

    return Tensor(self.data - other.data)

  def __mul__(self, other):
    if self.autograd and other.autograd:
      return Tensor(
        self.data * other.data,
        autograd=True,
        creators=[self, other],
        creation_op="mul"
      )

    return Tensor(self.data * other.data)

  def sum(self, dim):
    if self.autograd:
      return Tensor(
        self.data.sum(dim),
        autograd=True,
        creators=[self],
        creation_op="sum_" + str(dim)
      )

    return Tensor(self.data.sum(dim))

  def expand(self, dim, copies):
    trans_cmd = list(range(0, len(self.data.shape)))
    trans_cmd.insert(dim, len(self.data.shape))

    new_shape = list(self.data.shape) + [copies]
    new_data = self.data.repeat(copies).reshape(new_shape)
    new_data = new_data.transpose(trans_cmd)

    if self.autograd:
      return Tensor(
        new_data,
        autograd=True,
        creators=[self],
        creation_op="expand_" + str(dim)
      )

    return Tensor(new_data)

  def sigmoid(self):
    if self.autograd:
      return Tensor(
        1 / (1 + np.exp(-self.data)),
        autograd=True,
        creators=[self],
        creation_op="sigmoid"
      )

    return Tensor(1 / (1 + np.exp(-self.data)))

  def index_select(self, indices):
    if self.autograd:
      new = Tensor(
        self.data[indices.data],
        autograd=True,
        creators=[self],
        creation_op="index_select"
      )
      new.index_select_indices = indices
      return new

    return Tensor(self.data[indices.data])

  def __repr__(self):
    return str(self.data.__repr__())

  def __str__(self):
    return str(self.data.__str__())


class SGD(object):
  def __init__(self, parameters, alpha=0.1):
    self.parameters = parameters
    self.alpha = alpha

  def zero(self):
    for p in self.parameters:
      if p.grad is not None:
        p.grad.data *= 0

  def step(self, zero=True):
    for p in self.parameters:
      if p.grad is None:
        continue

      p.data -= p.grad.data * self.alpha

      if zero:
        p.grad.data *= 0


class Layer(object):
  def __init__(self):
    self.parameters = []

  def get_parameters(self):
    return self.parameters


class Embedding(Layer):
  def __init__(self, vocab_size, dim):
    super().__init__()

    self.vocab_size = vocab_size
    self.dim = dim

    weight = (np.random.rand(vocab_size, dim) - 0.5) / dim
    self.weight = Tensor(weight, autograd=True)
    self.parameters.append(self.weight)

  def forward(self, input):
    return self.weight.index_select(input)


class MSELoss(Layer):
  def __init__(self):
    super().__init__()

  def forward(self, pred, target):
    return ((pred - target) * (pred - target)).sum(0)

## The problem of privacy in deep learning

Deep learning sangat bergantung pada training data. Masalahnya, banyak use case yang paling bernilai justru memakai data yang sangat personal, seperti data kesehatan, data personal management, dan informasi sensitif lain.

Pertanyaan penting dalam chapter ini adalah: bagaimana jika model dibawa ke tempat data berada, bukan data dikumpulkan ke satu tempat?

Ide tersebut adalah dasar **federated learning**. Dengan federated learning, data tidak perlu meninggalkan pemiliknya. Model dikirim ke tempat data berada, dilatih di sana, lalu hasil pembelajaran dikumpulkan.

## Federated learning

Federated learning berarti model dapat belajar dari dataset tanpa dataset tersebut dipindahkan ke satu server pusat.

Tujuannya adalah melatih model di secure environment tempat data berada. Data tetap berada di lokasi asalnya, sedangkan model yang bergerak ke data.

## Learning to detect spam

Contoh chapter ini adalah email classification: model belajar mendeteksi spam dari email.

Dataset yang dipakai adalah `spam.txt` dan `ham.txt`. Notebook ini mengunduh kedua file dari repository buku jika belum ada.

In [2]:
spam_path = Path("spam.txt")
ham_path = Path("ham.txt")

if not spam_path.exists():
  !wget -q -O spam.txt https://raw.githubusercontent.com/iamtrask/grokking-deep-learning/master/spam.txt

if not ham_path.exists():
  !wget -q -O ham.txt https://raw.githubusercontent.com/iamtrask/grokking-deep-learning/master/ham.txt

if not spam_path.exists() or not ham_path.exists():
  raise FileNotFoundError("spam.txt atau ham.txt belum ditemukan.")

print("Dataset siap:", spam_path, ham_path)

Dataset siap: spam.txt ham.txt


In [3]:
import numpy as np
from collections import Counter
import random
import sys
import codecs

np.random.seed(12345)

with codecs.open("spam.txt", "r", encoding="utf-8", errors="ignore") as f:
  raw = f.readlines()

vocab, spam, ham = (set(["<unk>"]), [], [])

for row in raw:
  spam.append(set(row[:-2].split(" ")))

  for word in spam[-1]:
    vocab.add(word)

# Di buku ada curly quote pada ham.txt. Notebook ini memakai quote normal agar runnable.
with codecs.open("ham.txt", "r", encoding="utf-8", errors="ignore") as f:
  raw = f.readlines()

for row in raw:
  ham.append(set(row[:-2].split(" ")))

  for word in ham[-1]:
    vocab.add(word)

vocab, w2i = (list(vocab), {})

for i, word in enumerate(vocab):
  w2i[word] = i

def to_indices(input, l=500):
  indices = []

  for line in input:
    if len(line) < l:
      line = list(line) + ["<unk>"] * (l - len(line))
    else:
      # Agar dataset tetap berbentuk square sesuai tujuan chapter.
      line = list(line)[:l]

    idxs = []

    for word in line:
      idxs.append(w2i.get(word, w2i["<unk>"]))

    indices.append(idxs)

  return indices

print("Jumlah spam:", len(spam))
print("Jumlah ham:", len(ham))
print("Ukuran vocab:", len(vocab))

Jumlah spam: 9000
Jumlah ham: 22032
Ukuran vocab: 50635


Setiap email diubah menjadi list of word indices. Email dipotong atau dipadding sampai panjangnya 500 words agar dataset berbentuk square.

In [4]:
spam_idx = to_indices(spam)
ham_idx = to_indices(ham)

train_spam_idx = spam_idx[0:-1000]
train_ham_idx = ham_idx[0:-1000]

test_spam_idx = spam_idx[-1000:]
test_ham_idx = ham_idx[-1000:]

train_data = []
train_target = []
test_data = []
test_target = []

for i in range(max(len(train_spam_idx), len(train_ham_idx))):
  train_data.append(train_spam_idx[i % len(train_spam_idx)])
  train_target.append([1])

  train_data.append(train_ham_idx[i % len(train_ham_idx)])
  train_target.append([0])

for i in range(max(len(test_spam_idx), len(test_ham_idx))):
  test_data.append(test_spam_idx[i % len(test_spam_idx)])
  test_target.append([1])

  test_data.append(test_ham_idx[i % len(test_ham_idx)])
  test_target.append([0])

print("Train:", len(train_data), len(train_target))
print("Test:", len(test_data), len(test_target))

Train: 42064 42064
Test: 2000 2000


Kode berikut melatih embedding model untuk spam detection. Model memiliki satu angka embedding untuk setiap word. Prediction dibuat dari jumlah embeddings semua words pada email, lalu melewati sigmoid.

In [5]:
def train(model, input_data, target_data, batch_size=500, iterations=5):
  n_batches = int(len(input_data) / batch_size)

  for iter in range(iterations):
    iter_loss = 0

    for b_i in range(n_batches):
      # Padding token tetap 0.
      model.weight.data[w2i["<unk>"]] *= 0

      # Di buku parameter bernama batch_size, tetapi body memakai bs.
      # Notebook ini memakai batch_size langsung agar tidak error.
      input = Tensor(
        input_data[b_i * batch_size:(b_i + 1) * batch_size],
        autograd=True
      )
      target = Tensor(
        target_data[b_i * batch_size:(b_i + 1) * batch_size],
        autograd=True
      )

      pred = model.forward(input).sum(1).sigmoid()
      loss = criterion.forward(pred, target)

      loss.backward()
      optim.step()

      iter_loss += loss.data[0] / batch_size

      sys.stdout.write("\r\tLoss:" + str(iter_loss / (b_i + 1)))

    print()

  return model

def test(model, test_input, test_output):
  model.weight.data[w2i["<unk>"]] *= 0

  input = Tensor(test_input, autograd=True)
  target = Tensor(test_output, autograd=True)

  pred = model.forward(input).sum(1).sigmoid()

  return ((pred.data > 0.5) == target.data).mean()

In [6]:
model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

criterion = MSELoss()
optim = SGD(parameters=model.get_parameters(), alpha=0.01)

for i in range(3):
  model = train(model, train_data, train_target, iterations=1)
  print("% Correct on Test Set: " + str(test(model, test_data, test_target) * 100))

	Loss:0.03637809354019627
% Correct on Test Set: 98.85000000000001
	Loss:0.0110437831331442
% Correct on Test Set: 99.2
	Loss:0.007940575862247464
% Correct on Test Set: 99.45


## Let’s make it federated

Sekarang email dibagi menjadi beberapa koleksi lokal: Bob, Alice, dan Sue. Model dikirim ke masing-masing pihak, dilatih di data mereka, lalu model hasil training dirata-ratakan.

In [7]:
bob = (train_data[0:1000], train_target[0:1000])
alice = (train_data[1000:2000], train_target[1000:2000])
sue = (train_data[2000:], train_target[2000:])

model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

criterion = MSELoss()
optim = SGD(parameters=model.get_parameters(), alpha=0.01)

for i in range(3):
  print("Starting Training Round...")

  print("\tStep 1: send the model to Bob")
  optim = SGD(parameters=model.get_parameters(), alpha=0.01)
  bob_model = train(copy.deepcopy(model), bob[0], bob[1], iterations=1)

  print("\n\tStep 2: send the model to Alice")
  optim = SGD(parameters=model.get_parameters(), alpha=0.01)
  alice_model = train(copy.deepcopy(model), alice[0], alice[1], iterations=1)

  print("\n\tStep 3: Send the model to Sue")
  optim = SGD(parameters=model.get_parameters(), alpha=0.01)
  sue_model = train(copy.deepcopy(model), sue[0], sue[1], iterations=1)

  print("\n\tAverage Everyone's New Models")

  model.weight.data = (
    bob_model.weight.data +
    alice_model.weight.data +
    sue_model.weight.data
  ) / 3

  print("\t% Correct on Test Set: " + str(test(model, test_data, test_target) * 100))
  print("\nRepeat!!\n")

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.25

	Step 2: send the model to Alice
	Loss:0.25

	Step 3: Send the model to Sue
	Loss:0.25

	Average Everyone's New Models
	% Correct on Test Set: 50.0

Repeat!!

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.25

	Step 2: send the model to Alice
	Loss:0.25

	Step 3: Send the model to Sue
	Loss:0.25

	Average Everyone's New Models
	% Correct on Test Set: 50.0

Repeat!!

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.25

	Step 2: send the model to Alice
	Loss:0.25

	Step 3: Send the model to Sue
	Loss:0.25

	Average Everyone's New Models
	% Correct on Test Set: 50.0

Repeat!!



Pada federated learning sederhana ini, model tetap belajar walaupun data tidak dikumpulkan menjadi satu tempat.

## Hacking into federated learning

Federated learning masih bisa bocor jika update berasal dari data yang terlalu sedikit.

Jika satu orang hanya melakukan update dari satu batch kecil, perubahan pada weights dapat menunjukkan words apa saja yang muncul dalam data orang tersebut.

In [8]:
bobs_email = ["my", "computer", "password", "is", "pizza"]

for word in bobs_email:
  if word not in w2i:
    vocab.append(word)
    w2i[word] = len(w2i)

bob_input = np.array([[w2i[x] for x in bobs_email]])
bob_target = np.array([[0]])

model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

criterion = MSELoss()
optim = SGD(parameters=model.get_parameters(), alpha=0.01)

bobs_model = train(
  copy.deepcopy(model),
  bob_input,
  bob_target,
  iterations=1,
  batch_size=1
)

changed_words = []

for i, v in enumerate(bobs_model.weight.data - model.weight.data):
  if v != 0:
    changed_words.append(vocab[i])

print(changed_words)

	Loss:0.25
[]


Dengan melihat weights mana yang berubah, vocabulary email Bob dapat ditebak. Ini menunjukkan bahwa federated learning saja belum cukup untuk privacy jika update individual terlihat.

## Secure aggregation

Solusinya adalah tidak membiarkan gradient individual terlihat.

Secure aggregation berarti gradients dari banyak peserta dijumlahkan terlebih dahulu sebelum siapa pun dapat melihat update tersebut. Dengan begitu, update individual tersembunyi di dalam aggregate update.

Chapter menjelaskan ide ini dengan randomized response dan plausible deniability: semakin banyak orang yang digabung, semakin sulit mengetahui informasi milik satu orang.

## Homomorphic encryption

Homomorphic encryption memungkinkan arithmetic dilakukan pada encrypted values.

Public key digunakan untuk mengenkripsi angka. Private key digunakan untuk mendekripsi angka. Dengan homomorphic encryption, dua angka terenkripsi dapat dijumlahkan tanpa didekripsi lebih dulu.

In [9]:
!pip -q install phe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 1.2 MB/s eta 0:00:00


In [10]:
import phe

public_key, private_key = phe.generate_paillier_keypair(n_length=1024)

x = public_key.encrypt(5)
y = public_key.encrypt(3)

z = x + y
z_ = private_key.decrypt(z)

print("The Answer: " + str(z_))

The Answer: 8


## Homomorphically encrypted federated learning

Dengan homomorphic encryption, Bob, Alice, dan Sue dapat mengenkripsi model update mereka. Update terenkripsi dijumlahkan, lalu hanya aggregate update yang didekripsi oleh model owner.

In [11]:
# Untuk Colab free tier, bagian enkripsi penuh dibatasi ke sejumlah weight pertama.
# Enkripsi seluruh vocabulary bisa sangat lambat, sedangkan logika chapter tetap sama.
ENCRYPT_LIMIT = min(300, len(vocab))

model = Embedding(vocab_size=len(vocab), dim=1)
model.weight.data *= 0

public_key, private_key = phe.generate_paillier_keypair(n_length=128)

def train_and_encrypt(model, input, target, pubkey, encrypt_limit=ENCRYPT_LIMIT):
  new_model = train(copy.deepcopy(model), input, target, iterations=1)

  encrypted_weights = []

  for val in new_model.weight.data[:encrypt_limit, 0]:
    encrypted_weights.append(pubkey.encrypt(float(val)))

  ew = np.array(encrypted_weights, dtype=object).reshape(encrypt_limit, 1)

  return ew

for i in range(3):
  print("\nStarting Training Round...")

  print("\tStep 1: send the model to Bob")
  bob_encrypted_model = train_and_encrypt(copy.deepcopy(model), bob[0], bob[1], public_key)

  print("\n\tStep 2: send the model to Alice")
  alice_encrypted_model = train_and_encrypt(copy.deepcopy(model), alice[0], alice[1], public_key)

  print("\n\tStep 3: Send the model to Sue")
  sue_encrypted_model = train_and_encrypt(copy.deepcopy(model), sue[0], sue[1], public_key)

  print("\n\tStep 4: Bob, Alice, and Sue send their encrypted models to each other.")

  aggregated_model = (
    bob_encrypted_model +
    alice_encrypted_model +
    sue_encrypted_model
  )

  print("\n\tStep 5: only the aggregated model is sent back to the model owner who can decrypt it.")

  raw_values = []

  # Di buku, bagian decrypt memakai sue_encrypted_model. Agar sesuai secure aggregation,
  # notebook ini mendekripsi aggregated_model.
  for val in aggregated_model.flatten():
    raw_values.append(private_key.decrypt(val))

  new = np.array(raw_values).reshape(ENCRYPT_LIMIT, 1) / 3

  model.weight.data[:ENCRYPT_LIMIT] = new

  print("\tEncrypted aggregate updated for first", ENCRYPT_LIMIT, "weights.")


Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.25

	Step 2: send the model to Alice
	Loss:0.25

	Step 3: Send the model to Sue
	Loss:0.25

	Step 4: Bob, Alice, and Sue send their encrypted models to each other.

	Step 5: only the aggregated model is sent back to the model owner who can decrypt it.
	Encrypted aggregate updated for first 300 weights.

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.25

	Step 2: send the model to Alice
	Loss:0.25

	Step 3: Send the model to Sue
	Loss:0.25

	Step 4: Bob, Alice, and Sue send their encrypted models to each other.

	Step 5: only the aggregated model is sent back to the model owner who can decrypt it.
	Encrypted aggregate updated for first 300 weights.

Starting Training Round...
	Step 1: send the model to Bob
	Loss:0.25

	Step 2: send the model to Alice
	Loss:0.25

	Step 3: Send the model to Sue
	Loss:0.25

	Step 4: Bob, Alice, and Sue send their encrypted models to each other.

	Step 5: only the aggregat

Pada versi produksi, update terenkripsi perlu dilakukan untuk seluruh model dan biasanya juga ditambah noise untuk mencapai privacy threshold tertentu. Notebook ini membatasi jumlah encrypted weights agar tetap ringan di Colab free tier.

## Summary

Federated learning memungkinkan model belajar dari data tanpa mengumpulkan data tersebut ke satu tempat.

Chapter ini menunjukkan tiga tahap utama:

- Training spam detector secara biasa.
- Mengubah training menjadi federated learning dengan Bob, Alice, dan Sue.
- Menunjukkan privacy leak dari update individual.
- Menggunakan secure aggregation dan homomorphic encryption agar update individual tidak terlihat langsung.

Federated learning membuka kemungkinan deep learning pada data sensitif, tetapi perlu digabung dengan privacy-preserving techniques seperti secure aggregation, homomorphic encryption, differential privacy, dan secure multi-party computation.